# Olist E-Commerce: SQL Queries used for Tableau Public Dashboards
**Dataset**: [Brazilian E-commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)   
**Layer Used**: Gold: Business-ready analytics tables   
**Tools Used**: Databricks, Spark SQL, Delta Lake

This notebook documents the SQL queries used to power the Tableau Public dashboards included in this repository. The dashboards are built exclusively from curated Gold-layer analytics tables produced through the Medallion Architecture ETL pipeline.

## Workflow:

Raw Olist Dataset

        ↓
Bronze Layer
(raw ingestion)

        ↓
Silver Layer
(cleaned & validated)

        ↓
Gold Layer
(star schema fact & dimension tables)

        ↓
Spark SQL Queries
(this notebook)

        ↓
Tableau Public Dashboards

## Gold Layer Table References

The SQL queries demonstrate how the Gold-layer data model supports business reporting across product performance, seller analytics, regional trends, customer behavior, and executive KPI reporting.

| Table | Grain | Description |
|---|---|---|
| `gold_fact_order_items` | One row per order line item | Item-level revenue, freight, and delivery metrics |
| `gold_fact_orders` | One row per order | Order-level revenue, payment, review, and delivery summary |
| `gold_dim_customers` | One row per customer_id | Customer geography and region |
| `gold_dim_products` | One row per product | Product catalogue with dimensions and macro-category |
| `gold_dim_sellers` | One row per seller | Seller geography and region |
| `gold_dim_dates` | One row per calendar date | Date intelligence: year, month, quarter, holidays, weekday flags |

## Product Performance Queries

These queries support Tableau dashboards exploring product sales, pricing, freight costs, seasonality, and delivery performance

In [0]:
%sql
SELECT
    order_status,
    COUNT(*)                                AS order_count,
    ROUND(COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (), 2)         AS pct_of_total,
    ROUND(AVG(order_total_value), 2)        AS avg_order_value,
    ROUND(SUM(order_total_value), 2)        AS total_value,
    ROUND(AVG(days_to_deliver), 1)          AS avg_days_to_deliver
FROM gold_fact_orders
GROUP BY order_status
ORDER BY order_count DESC;
--Feeds: "Olist Sales Performance & Fulfillment Analysis" Dashboard, "Order Status Funnel" bar chart

order_status,order_count,pct_of_total,avg_order_value,total_value,avg_days_to_deliver
Delivered,96469,97.77,159.83,1.541825137E7,12.5
Shipped,1106,1.12,160.15,177129.34,null
Canceled,461,0.47,229.69,105885.72,20.3
Invoiced,312,0.32,221.12,68988.75,null
In-Progress,303,0.31,229.82,69635.19,null
Unavailable,14,0.01,251.39,3519.41,null


In [0]:
%sql
WITH order_level AS (
    SELECT
        order_id,
        days_to_deliver,
        delivered_on_time,
        order_total_value,
        review_score,
        order_status
    FROM gold_fact_orders
    WHERE is_delivered = TRUE
      AND days_to_deliver IS NOT NULL
      AND days_to_deliver >= 0
)
SELECT
    p.product_category,
    p.weight_category,
    p.size_category,
    o.order_id,
    o.days_to_deliver,
    o.delivered_on_time,
    o.review_score,
    oi.item_price,
    oi.item_freight,
    ROUND(
        oi.item_freight
        / NULLIF(oi.item_total_value, 0),
    3) AS freight_ratio,
    CASE
        WHEN o.days_to_deliver <= 7  THEN '1 - Fast (≤7 days)'
        WHEN o.days_to_deliver <= 14 THEN '2 - Standard (8–14 days)'
        WHEN o.days_to_deliver <= 30 THEN '3 - Slow (15–30 days)'
        ELSE                              '4 - Very Slow (>30 days)'
    END                                         AS delivery_speed_band,
    CASE
        WHEN oi.item_freight
             / NULLIF(oi.item_total_value, 0) >= 0.30 THEN 'High ≥30%'
        WHEN oi.item_freight
             / NULLIF(oi.item_total_value, 0) >= 0.20 THEN 'Moderate 20–30%'
        ELSE                                                'Healthy <20%'
    END                                         AS freight_burden_flag

FROM gold_fact_order_items oi
JOIN order_level o          ON oi.order_id = o.order_id
JOIN gold_dim_products p    ON oi.product_id = p.product_id
WHERE p.product_category IS NOT NULL
  AND p.product_category != 'Unknown';
--Feeds: "Olist Sales Performance & Fulfillment Analysis" Dashboard, "Order Delivery Distribution by Product Category" box plot

product_category,weight_category,size_category,order_id,days_to_deliver,delivered_on_time,review_score,item_price,item_freight,freight_ratio,delivery_speed_band,freight_burden_flag
Fashion & Accessories,Light,Small,00063b381e2406b52ad429470734ebd5,11,true,5.0,45.0,12.98,0.224,2 - Standard (8–14 days),Moderate 20–30%
Living & Home Appliances,Medium,Medium,0006ec9db01a64e59a68b2c340bf65a7,7,true,5.0,74.0,23.32,0.24,1 - Fast (≤7 days),Moderate 20–30%
Office & Business,Medium,Medium,000f25f4d72195062c040b12dce9a18a,15,true,4.0,119.99,44.4,0.27,3 - Slow (15–30 days),Moderate 20–30%
"Automotive, Construction & Gardening",Medium,Small,0016dfedd97fc2950e388d2971d718c7,24,true,5.0,49.75,20.8,0.295,3 - Slow (15–30 days),Moderate 20–30%
Electronics & Gaming,Light,Small,001e7ba991be1b19605ca0316e7130f9,10,true,5.0,195.0,18.21,0.085,2 - Standard (8–14 days),Healthy <20%
Electronics & Gaming,Light,Small,00259a44fcad3fc0474329e925d14fc3,8,true,4.0,19.99,14.1,0.414,2 - Standard (8–14 days),High ≥30%
"Health, Beauty & Personal Care",Light,Small,002d040018d12a3853c059f7f23ab5b1,8,true,3.0,155.0,14.84,0.087,2 - Standard (8–14 days),Healthy <20%
"Automotive, Construction & Gardening",Medium,Medium,00324b3eda39ba5ecce3945823e3594c,23,false,5.0,76.0,34.07,0.31,3 - Slow (15–30 days),High ≥30%
Living & Home Appliances,Medium,Medium,0032d07457ae9c806c79368d7d9ce96b,50,false,1.0,159.0,27.19,0.146,4 - Very Slow (>30 days),Healthy <20%
Electronics & Gaming,Light,Small,00584d79015452d19fd3d044ba3abdc0,3,true,5.0,109.9,12.27,0.1,1 - Fast (≤7 days),Healthy <20%


In [0]:
%sql
SELECT
    DATE_FORMAT(order_purchase_date, 'MMM-yyyy')    AS year_month,
    p.product_category                               AS product_category,
    ROUND(SUM(oi.item_total_value), 0)              AS monthly_revenue
FROM gold_fact_order_items oi
JOIN gold_dim_products p    ON oi.product_id = p.product_id
GROUP BY
    DATE_FORMAT(order_purchase_date, 'MMM-yyyy'),
    p.product_category
ORDER BY
    MIN(order_purchase_date),
    p.product_category;
--Feeds: "Olist Sales Performance & Fulfillment Analysis" Dashboard, Monthly Sales Revenue for Top Product Categories area chart

year_month,product_category,monthly_revenue
Sep-2016,Living & Home Appliances,136.0
Sep-2016,Electronics & Gaming,75.0
Sep-2016,"Health, Beauty & Personal Care",143.0
Oct-2016,Pet & Baby,2595.0
Oct-2016,Fashion & Accessories,803.0
Oct-2016,Gifts & Misc,3468.0
Oct-2016,Living & Home Appliances,11106.0
Oct-2016,"Sports, Toys & Leisure",10240.0
Oct-2016,"Automotive, Construction & Gardening",4156.0
Oct-2016,Electronics & Gaming,8769.0


In [0]:
%sql
SELECT
    p.product_category,
    COUNT(DISTINCT oi.order_id)                             AS total_orders,
    ROUND(SUM(oi.item_price), 2)                           AS total_item_revenue,
    ROUND(SUM(oi.item_freight), 2)                         AS total_freight,
    ROUND(SUM(oi.item_freight)
        / NULLIF(SUM(oi.item_total_value), 0), 3)          AS freight_ratio,
    ROUND(AVG(oi.item_price), 2)                           AS avg_item_price,
    ROUND(AVG(oi.item_freight), 2)                         AS avg_freight,
    ROUND(AVG(p.product_weight_kg), 2)                     AS avg_weight_kg,
    ROUND(AVG(o.days_to_deliver), 1)                       AS avg_delivery_days
FROM gold_fact_order_items oi
JOIN gold_dim_products p ON oi.product_id = p.product_id
JOIN gold_fact_orders o  ON oi.order_id = o.order_id
GROUP BY p.product_category
ORDER BY freight_ratio DESC;
--Feeds: "Olist Freight Burden Analysis" Dashboard, "Freight Burden by Product Category" bar chart

product_category,total_orders,total_item_revenue,total_freight,freight_ratio,avg_item_price,avg_freight,avg_weight_kg,avg_delivery_days
Security & Utilities,142,21792.52,6549.04,0.231,108.42,32.58,3.86,10.5
Food & Beverage,974,67001.59,17520.27,0.207,57.41,15.01,0.89,10.1
Office & Business,3864,533282.4,120826.49,0.185,118.01,26.74,5.87,15.7
Fashion & Accessories,3441,342648.66,72178.09,0.174,91.76,19.33,2.01,11.4
Living & Home Appliances,24776,3090605.97,620313.01,0.167,103.37,20.75,2.83,12.3
Books & Media,933,83356.53,16452.67,0.165,83.69,16.52,0.84,11.1
Pet & Baby,4595,626080.3,107914.35,0.147,124.92,21.53,3.14,11.9
"Automotive, Construction & Gardening",9748,1541186.72,258682.41,0.144,134.79,22.62,2.83,12.5
Electronics & Gaming,15383,1902576.26,307740.73,0.139,110.16,17.82,0.76,13.0
Unknown,1451,179535.28,28169.81,0.136,112.0,17.57,1.65,12.7


In [0]:
%sql
SELECT
    p.product_category,
    p.product_id,
    ROUND(AVG(oi.item_price), 2)                                AS avg_price,
    ROUND(AVG(oi.item_freight), 2)                              AS avg_freight,
    ROUND(AVG(oi.item_freight)
        / NULLIF(AVG(oi.item_total_value), 0), 3)               AS avg_freight_ratio,
    COUNT(DISTINCT oi.order_id)                                 AS times_ordered,
    ROUND(SUM(oi.item_total_value), 2)                          AS total_revenue,
    ROUND(AVG(p.product_weight_kg), 2)                          AS avg_weight_kg
FROM gold_fact_order_items oi
JOIN gold_dim_products p ON oi.product_id = p.product_id
GROUP BY p.product_category, p.product_id
HAVING COUNT(DISTINCT oi.order_id) >= 5;
--Feeds: "Olist Freight Burden Analysis" Dashboard, individual product freight risk scatterplot

product_category,product_id,avg_price,avg_freight,avg_freight_ratio,times_ordered,total_revenue,avg_weight_kg
Electronics & Gaming,381169576b9f3c083a1c3261da161a3b,46.86,18.81,0.286,8,525.39,0.23
Fashion & Accessories,5637032095493f280eae8d9da00da6f1,49.99,13.51,0.213,7,507.96,0.75
"Automotive, Construction & Gardening",422879e10f46682990de24d770e7f83d,54.91,15.75,0.223,352,34201.26,1.55
"Health, Beauty & Personal Care",2b4609f8948be18874494203496bc318,87.37,15.16,0.148,259,26659.74,0.25
Electronics & Gaming,a00a22560435cc439faf6163eabc1ef8,18.99,16.78,0.469,6,214.61,0.08
Living & Home Appliances,0efe972876d70927f5099b77f387c6c7,83.9,14.64,0.149,7,689.78,1.35
"Automotive, Construction & Gardening",c0aed8767e46cce56e77db0ceef83035,79.9,17.66,0.181,10,1073.13,0.5
Living & Home Appliances,06edb72f1e0c64b14c5b79353f7abea3,40.78,14.87,0.267,130,7958.33,0.35
Living & Home Appliances,2635c3e7db0ac6cb3e733cac61ce0ba5,26.9,17.15,0.389,15,704.75,0.63
Living & Home Appliances,e19ddcc85537b41f22116c8d5425ef46,29.99,11.85,0.283,18,878.72,0.7


In [0]:
%sql
WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('month', order_purchase_date) AS month_date,
        YEAR(order_purchase_date) AS purchase_year,
        MONTH(order_purchase_date) AS purchase_month,
        product_id,
        product_category,
        COUNT(*) AS units_sold,
        ROUND(AVG(item_price), 2) AS avg_item_cost,
        ROUND(AVG(item_freight), 2) AS avg_freight_value,
        ROUND(SUM(item_freight) / NULLIF(SUM(item_total_value), 0), 2)
            AS freight_cost_ratio_to_total_value
    FROM gold_fact_order_items
    GROUP BY
        DATE_TRUNC('month', order_purchase_date),
        YEAR(order_purchase_date),
        MONTH(order_purchase_date),
        product_id,
        product_category
),

products AS (
    SELECT DISTINCT product_id
    FROM gold_fact_order_items
),

months AS (
    SELECT DISTINCT DATE_TRUNC('month', order_purchase_date) AS month_date
    FROM gold_fact_order_items
),

product_month_grid AS (
    SELECT
        p.product_id,
        m.month_date
    FROM products p
    CROSS JOIN months m
),

complete_monthly_sales AS (
    SELECT
        g.product_id,
        g.month_date,
        YEAR(g.month_date) AS purchase_year,
        MONTH(g.month_date) AS purchase_month,
        COALESCE(s.units_sold, 0) AS units_sold,
        s.product_category,
        COALESCE(s.avg_item_cost, 0) AS avg_item_cost,
        COALESCE(s.avg_freight_value, 0) AS avg_freight_value,
        COALESCE(s.freight_cost_ratio_to_total_value, 0)
            AS freight_cost_ratio_to_total_value
    FROM product_month_grid g
    LEFT JOIN monthly_sales s
        ON g.product_id = s.product_id
       AND g.month_date = s.month_date
),

rolling_metrics AS (
    SELECT
        *,
        SUM(units_sold) OVER (
            PARTITION BY product_id
            ORDER BY month_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS current_3_month_units,

        SUM(units_sold) OVER (
            PARTITION BY product_id
            ORDER BY month_date
            ROWS BETWEEN 5 PRECEDING AND 3 PRECEDING
        ) AS previous_3_month_units
    FROM complete_monthly_sales
),

final_metrics AS (
    SELECT
        *,
        CASE
            WHEN previous_3_month_units IS NULL
                OR previous_3_month_units = 0
            THEN NULL
            ELSE ROUND(
                100.0 *
                (current_3_month_units - previous_3_month_units)
                / previous_3_month_units,
                2
            )
        END AS rolling_3_month_growth
    FROM rolling_metrics
),

top_flagged_products AS (
    SELECT product_id
    FROM (
        SELECT
            product_id,
            MAX(
                0.4 * freight_cost_ratio_to_total_value
                +
                0.4 * GREATEST(
                    0,
                    (previous_3_month_units - current_3_month_units)
                    / NULLIF(previous_3_month_units, 0)
                )
                +
                0.2 * (
                    1.0 / LOG(1 + current_3_month_units)
                )
            ) AS max_discontinuation_score
        FROM final_metrics
        WHERE
            freight_cost_ratio_to_total_value >= 0.30
            AND current_3_month_units IS NOT NULL
            AND previous_3_month_units IS NOT NULL
            AND rolling_3_month_growth <= -40
            AND (current_3_month_units + previous_3_month_units) >= 10
        GROUP BY product_id
        ORDER BY max_discontinuation_score DESC,
                 product_id
        LIMIT 20
    ) flagged
)

SELECT
    fm.product_id,
    fm.product_category,
    fm.month_date,
    fm.purchase_year,
    fm.purchase_month,
    fm.units_sold,
    fm.current_3_month_units,
    fm.previous_3_month_units,
    fm.rolling_3_month_growth,
    fm.avg_item_cost,
    fm.avg_freight_value,
    fm.freight_cost_ratio_to_total_value
FROM final_metrics fm
JOIN top_flagged_products tfp
    ON fm.product_id = tfp.product_id
ORDER BY
    fm.product_id,
    fm.month_date;
--Feeds: "Olist Freight Burden Analysis" Dashboard, "Pipeline Risk Products" table

product_id,product_category,month_date,purchase_year,purchase_month,units_sold,current_3_month_units,previous_3_month_units,rolling_3_month_growth,avg_item_cost,avg_freight_value,freight_cost_ratio_to_total_value
0152f69b6cf919bcdaf117aa8c43e5a2,null,2016-09-01T00:00:00.000Z,2016,9,0,0,null,null,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,null,2016-10-01T00:00:00.000Z,2016,10,0,0,null,null,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,null,2016-12-01T00:00:00.000Z,2016,12,0,0,null,null,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,null,2017-01-01T00:00:00.000Z,2017,1,0,0,0,null,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,Living & Home Appliances,2017-02-01T00:00:00.000Z,2017,2,1,1,0,null,13.9,14.52,0.51
0152f69b6cf919bcdaf117aa8c43e5a2,Living & Home Appliances,2017-03-01T00:00:00.000Z,2017,3,15,16,0,null,13.9,12.66,0.48
0152f69b6cf919bcdaf117aa8c43e5a2,Living & Home Appliances,2017-04-01T00:00:00.000Z,2017,4,11,27,0,null,13.9,14.57,0.51
0152f69b6cf919bcdaf117aa8c43e5a2,null,2017-05-01T00:00:00.000Z,2017,5,0,26,1,2500.00,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,null,2017-06-01T00:00:00.000Z,2017,6,0,11,16,-31.25,0.0,0.0,0.0
0152f69b6cf919bcdaf117aa8c43e5a2,null,2017-07-01T00:00:00.000Z,2017,7,0,0,27,-100.00,0.0,0.0,0.0


## Seller Performance Queries

Supports seller performance dashboards measuring revenue concentration, operational performance, delivery metrics, and seller productivity.

In [0]:
%sql
WITH seller_revenue AS (
    SELECT
        oi.seller_id,
        ROUND(SUM(oi.item_total_value), 2) AS total_revenue,
        s.seller_region,
        s.seller_state,
        ROUND(
            SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(DISTINCT oi.order_id), 0),
        1)                                                              AS on_time_pct,
        ROUND(AVG(o.review_score), 2)                                   AS avg_review_score
    FROM gold_fact_order_items oi
    JOIN gold_dim_sellers s ON
        oi.seller_id = s.seller_id
    JOIN gold_fact_orders o ON 
        oi.order_id = o.order_id
    GROUP BY oi.seller_id, s.seller_region, s.seller_state
),
ranked AS (
    SELECT
        *,
        ROUND(
            SUM(total_revenue) OVER (ORDER BY total_revenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            * 100.0
            / SUM(total_revenue) OVER (),
        2) AS cumulative_revenue_pct,
        ROW_NUMBER() OVER (ORDER BY total_revenue DESC) AS seller_rank
    FROM seller_revenue
),
scored AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY total_revenue ASC)      AS revenue_quartile,
        NTILE(4) OVER (ORDER BY on_time_pct ASC)        AS delivery_quartile,
        NTILE(4) OVER (ORDER BY avg_review_score ASC)   AS satisfaction_quartile
    FROM ranked
)
SELECT
    *,
    revenue_quartile + delivery_quartile + satisfaction_quartile    AS composite_score,
    CASE
        WHEN revenue_quartile = 4 AND delivery_quartile >= 3
             AND satisfaction_quartile >= 3                         THEN 'Star Seller'
        WHEN revenue_quartile >= 3 AND delivery_quartile <= 2       THEN 'High Revenue / Poor Delivery'
        WHEN revenue_quartile <= 2 AND satisfaction_quartile = 4    THEN 'Low Revenue / High Satisfaction'
        WHEN revenue_quartile <= 2 AND delivery_quartile <= 2       THEN 'Underperformer'
        ELSE                                                             'Developing'
    END AS seller_tier
FROM scored
WHERE seller_rank <= 150
ORDER BY seller_rank;
--Feeds: "Seller Revenue Concentration and Platform Growth" Dashboard, Revenue Dependency bar chart

seller_id,total_revenue,seller_region,seller_state,on_time_pct,avg_review_score,cumulative_revenue_pct,seller_rank,revenue_quartile,delivery_quartile,satisfaction_quartile,composite_score,seller_tier
4869f7a5dfa277a7dca6462dcf3b52b2,249640.7,Southeast,SP,90.7,4.12,1.58,1,4,1,2,7,High Revenue / Poor Delivery
7c67e1448b00f6e969d365cea6b010ab,239536.44,Southeast,SP,125.8,3.34,3.09,2,4,4,1,9,Developing
53243585a1d6dc2643021fd1853d8905,235856.68,Northeast,BA,108.4,4.08,4.58,3,4,4,2,10,Developing
4a3ca9315b744ce9f8e9374361493884,235539.96,Southeast,SP,97.5,3.8,6.06,4,4,2,2,8,High Revenue / Poor Delivery
fa1c13f2614d7b5c4749cbc52fecda94,204084.73,Southeast,SP,89.9,4.34,7.35,5,4,1,3,8,High Revenue / Poor Delivery
da8622b14eb17ae2831f4ac5b9dab84a,185192.32,Southeast,SP,110.3,4.06,8.52,6,4,4,2,10,Developing
7e93a43ef30c4f03f38b393420bc753a,182754.05,Southeast,SP,91.4,4.21,9.67,7,4,1,3,8,High Revenue / Poor Delivery
1025f0e2d44d7041d6cf58b6550e0bfa,172860.69,Southeast,SP,144.8,3.85,10.76,8,4,4,2,10,Developing
7a67c85e85bb2ce8582c35f2203ad736,162648.38,Southeast,SP,94.4,4.23,11.79,9,4,2,3,9,High Revenue / Poor Delivery
955fee9216a65b617aa5c0531780ce60,160602.68,Southeast,SP,107.0,4.05,12.8,10,4,4,2,10,Developing


In [0]:
%sql
WITH order_level AS (
    SELECT
        order_id,
        customer_id,
        days_to_deliver,
        delivered_on_time,
        review_score
    FROM gold_fact_orders
    WHERE is_delivered = TRUE
),
seller_metrics AS (
    SELECT
        oi.seller_id,
        s.seller_state,
        s.seller_region,
        COUNT(DISTINCT oi.order_id)                                         AS total_orders,
        ROUND(SUM(oi.item_total_value), 2)                                  AS total_revenue,
        ROUND(AVG(oi.item_price), 2)                                        AS avg_item_price,
        COUNT(DISTINCT oi.product_id)                                       AS unique_products,
        ROUND(SUM(oi.item_total_value), 2)
            / NULLIF(COUNT(DISTINCT oi.order_id), 0)                        AS avg_revenue_per_order,
        ROUND(AVG(o.days_to_deliver), 1)                                    AS avg_delivery_days,
        ROUND(
            COUNT(DISTINCT CASE WHEN o.delivered_on_time
                THEN oi.order_id END) * 100.0
            / NULLIF(COUNT(DISTINCT oi.order_id), 0), 1)                    AS on_time_pct,
        ROUND(AVG(o.review_score), 2)                                       AS avg_review_score,
        COUNT(DISTINCT o.customer_id)                                       AS unique_customers
    FROM gold_fact_order_items oi
    JOIN order_level o      ON oi.order_id = o.order_id
    JOIN gold_dim_sellers s  ON oi.seller_id = s.seller_id
    GROUP BY oi.seller_id, s.seller_state, s.seller_region
    HAVING COUNT(DISTINCT oi.order_id) >= 10
),
scored AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY total_revenue ASC)      AS revenue_quartile,
        NTILE(4) OVER (ORDER BY on_time_pct ASC)        AS delivery_quartile,
        NTILE(4) OVER (ORDER BY avg_review_score ASC)   AS satisfaction_quartile
    FROM seller_metrics
)
SELECT
    seller_id,
    seller_state,
    seller_region,
    total_orders,
    total_revenue,
    avg_item_price,
    unique_products,
    avg_revenue_per_order,
    avg_delivery_days,
    on_time_pct,
    avg_review_score,
    unique_customers,
    revenue_quartile,
    delivery_quartile,
    satisfaction_quartile,
    revenue_quartile + delivery_quartile + satisfaction_quartile            AS composite_score,
    CASE
        WHEN revenue_quartile = 4 AND delivery_quartile >= 3
             AND satisfaction_quartile >= 3                                 THEN '1 - Star Seller'
        WHEN revenue_quartile >= 3 AND delivery_quartile <= 2               THEN '2 - High Rev / Poor Delivery'
        WHEN revenue_quartile <= 2 AND satisfaction_quartile = 4            THEN '3 - Low Rev / High Satisfaction'
        WHEN revenue_quartile <= 2 AND delivery_quartile <= 2               THEN '4 - Underperformer'
        ELSE                                                                     '5 - Developing'
    END AS seller_tier,
    ROUND((on_time_pct * 0.6) + (avg_review_score * 8.0), 1)               AS delivery_health_score
FROM scored
ORDER BY composite_score DESC, total_revenue DESC;
--Feeds "Seller Revenue Concentration and Platform Growth" Dashboard, "Revenue Concentration by Seller" heatmap AND
--"Seller Performance Scorecard" Dashboard, "Satisfaction vs. Delivery Performance" scatterplot AND
--"Seller Performance Scorecard" Dashboard, "Tier Performance Comparison" bar charts

seller_id,seller_state,seller_region,total_orders,total_revenue,avg_item_price,unique_products,avg_revenue_per_order,avg_delivery_days,on_time_pct,avg_review_score,unique_customers,revenue_quartile,delivery_quartile,satisfaction_quartile,composite_score,seller_tier,delivery_health_score
e882b2a25a10b9c057cc49695f222c19,RJ,Southeast,57,53913.76,880.3,19,945.8554385964912,9.4,100.0,4.59,57,4,4,4,12,1 - Star Seller,96.7
c3cfdc648177fdbbbb35635a37472c53,PR,South,278,49127.81,140.22,111,176.71874100719424,9.6,98.6,4.45,278,4,4,4,12,1 - Star Seller,94.8
12b9676b00f60f3b700e83af21824c0e,RS,South,133,29901.67,198.32,37,224.8245864661654,15.0,99.2,4.54,133,4,4,4,12,1 - Star Seller,95.8
612170e34b97004b3ba37eae81836b4c,RS,South,107,25372.08,209.68,48,237.12224299065423,10.8,99.1,4.43,107,4,4,4,12,1 - Star Seller,94.9
2b3e4a2a3ea8e01938cabda2a3e5cc79,SP,Southeast,52,24535.1,437.58,13,471.82884615384614,8.6,100.0,4.39,52,4,4,4,12,1 - Star Seller,95.1
744dac408745240a2c2528fb1b6028f3,PR,South,78,24200.4,268.95,75,310.26153846153846,11.2,98.7,4.55,78,4,4,4,12,1 - Star Seller,95.6
c72de06d72748d1a0dfb2125be43ba63,BA,Northeast,21,18733.18,796.45,8,892.0561904761905,9.7,100.0,4.45,21,4,4,4,12,1 - Star Seller,95.6
8ae520247981aa06bc94abddf5f46d34,SC,South,65,17766.34,251.09,50,273.3283076923077,10.9,100.0,4.45,65,4,4,4,12,1 - Star Seller,95.6
d921b68bf747894be13a97ae52b0f386,MG,Southeast,77,17403.81,203.19,46,226.02350649350652,9.9,98.7,4.56,77,4,4,4,12,1 - Star Seller,95.7
080199a181c46c657dc5aa235411be3b,SP,Southeast,79,15544.88,174.75,40,196.7706329113924,8.0,98.7,4.61,79,4,4,4,12,1 - Star Seller,96.1


In [0]:
%sql
SELECT
    DATE_FORMAT(DATE(o.order_date_key), 'yyyy-MM')   AS yr_month,
    COUNT(DISTINCT oi.seller_id)                     AS active_sellers,
    COUNT(DISTINCT oi.order_id)                      AS total_orders,
    ROUND(SUM(oi.item_total_value), 2)               AS total_revenue,
    ROUND(AVG(oi.item_total_value), 2)               AS avg_item_value,
    COUNT(DISTINCT oi.order_id)
        / NULLIF(COUNT(DISTINCT oi.seller_id), 0)    AS orders_per_seller
FROM gold_fact_order_items oi
JOIN gold_fact_orders o ON oi.order_id = o.order_id
GROUP BY DATE_FORMAT(DATE(o.order_date_key), 'yyyy-MM')
ORDER BY yr_month;
--Feeds "Seller Revenue Concentration and Platform Growth" Dashboard, "Platform Growth Trajectory" line graph

yr_month,active_sellers,total_orders,total_revenue,avg_item_value,orders_per_seller
2016-09,2,2,211.29,70.43,1.0
2016-10,143,308,56808.84,156.5,2.1538461538461537
2016-12,1,1,19.62,19.62,1.0
2017-01,227,789,137188.49,143.65,3.4757709251101323
2017-02,427,1733,286280.62,146.74,4.058548009367682
2017-03,499,2641,432048.59,144.02,5.292585170340681
2017-04,506,2391,412422.24,153.66,4.725296442687747
2017-05,583,3660,586190.95,141.73,6.2778730703259
2017-06,539,3217,502963.04,140.37,5.968460111317254
2017-07,606,3969,584971.62,129.45,6.5495049504950495


## Regional Opportunities Queries

Supports geographic dashboards comparing revenue, delivery performance, and customer satisfaction across regions and states in Brazil.

In [0]:
%sql
WITH order_revenue AS (
    SELECT
        c.customer_state,
        c.customer_region,
        COUNT(DISTINCT o.order_id)                                      AS total_orders,
        ROUND(SUM(o.order_total_value), 2)                              AS total_revenue,
        ROUND(AVG(o.order_total_value), 2)                              AS avg_order_value,
        ROUND(AVG(o.days_to_deliver), 1)                                AS avg_delivery_days,
        ROUND(
            SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
        1)                                                              AS on_time_pct,
        ROUND(AVG(o.review_score), 2)                                   AS avg_review_score
    FROM gold_fact_orders o
    JOIN gold_dim_customers c ON o.customer_id = c.customer_id
    WHERE o.is_delivered = TRUE
    GROUP BY c.customer_state, c.customer_region
),
seller_coverage AS (
    SELECT
        seller_state,
        COUNT(seller_id)                                     AS sellers_serving_state
    FROM gold_dim_sellers
    GROUP BY seller_state
)
SELECT
    r.customer_state,
    r.customer_region,
    r.total_orders,
    r.total_revenue,
    r.avg_order_value,
    r.avg_delivery_days,
    r.on_time_pct,
    r.avg_review_score,
    COALESCE(sc.sellers_serving_state, 0)                               AS sellers_serving_state
FROM order_revenue r
LEFT JOIN seller_coverage sc ON r.customer_state = sc.seller_state
ORDER BY r.avg_order_value DESC;
--Feeds "Regional Sales Landscape" Dashboard, "Revenue Concentration by State" choropleth map AND "Seller Coverage vs, Revenue by State" scatterplot

customer_state,customer_region,total_orders,total_revenue,avg_order_value,avg_delivery_days,on_time_pct,avg_review_score,sellers_serving_state
PB,Northeast,517,137838.55,266.61,20.4,89.6,4.08,6
AC,North,80,19575.33,244.69,21.0,96.3,4.09,1
AP,North,67,16141.81,240.92,27.2,97.0,4.24,0
AL,Northeast,397,94172.49,237.21,24.5,78.6,3.85,0
RO,North,243,56966.0,234.43,19.3,97.1,4.17,2
PA,North,946,212023.57,224.13,23.7,88.8,3.91,1
PI,Northeast,476,105178.19,220.96,19.4,86.1,3.99,1
RR,North,41,9039.52,220.48,29.3,87.8,3.9,0
TO,North,274,60007.37,219.01,17.6,90.1,4.15,0
RN,Northeast,474,100714.78,212.48,19.2,90.7,4.15,5


In [0]:
%sql
WITH national_avg AS (
    SELECT
        ROUND(
            SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
        1) AS national_on_time_pct
    FROM gold_fact_orders o
    WHERE o.is_delivered = TRUE
)
SELECT
    c.customer_state,
    c.customer_region,
    ROUND(
        SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
    1)                                          AS state_on_time_pct,
    n.national_on_time_pct,
    ROUND(
        SUM(CASE WHEN o.delivered_on_time THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(CASE WHEN o.is_delivered THEN 1 ELSE 0 END), 0),
    1) - n.national_on_time_pct                AS gap_vs_national_avg,
    COUNT(DISTINCT o.order_id)                  AS total_orders,
    ROUND(AVG(o.days_to_deliver), 1)            AS avg_delivery_days
FROM gold_fact_orders o
JOIN gold_dim_customers c ON o.customer_id = c.customer_id
CROSS JOIN national_avg n
WHERE o.is_delivered = TRUE
GROUP BY c.customer_state, c.customer_region, n.national_on_time_pct
ORDER BY gap_vs_national_avg ASC;
--Feeds "Regional Sales Landscape" Dashboard, Delivery Time Gap bar chart

customer_state,customer_region,state_on_time_pct,national_on_time_pct,gap_vs_national_avg,total_orders,avg_delivery_days
AL,Northeast,78.6,93.2,-14.6,397,24.5
MA,Northeast,82.6,93.2,-10.6,717,21.5
SE,Northeast,84.8,93.2,-8.4,335,21.5
PI,Northeast,86.1,93.2,-7.1,476,19.4
CE,Northeast,86.2,93.2,-7.0,1279,21.2
RR,North,87.8,93.2,-5.4,41,29.3
BA,Northeast,87.8,93.2,-5.4,3256,19.3
RJ,Southeast,87.9,93.2,-5.3,12350,15.2
PA,North,88.8,93.2,-4.4,946,23.7
ES,Southeast,89.3,93.2,-3.9,1995,15.7


In [0]:
%sql
SELECT
    d.year_month,
    d.year,
    d.month,
    c.customer_region,
    ROUND(SUM(o.order_total_value), 2)  AS monthly_revenue,
    COUNT(DISTINCT o.order_id)          AS monthly_orders,
    ROUND(AVG(o.review_score), 2)       AS avg_review_score
FROM gold_fact_orders o
JOIN gold_dim_customers c ON o.customer_id = c.customer_id
JOIN gold_dim_dates d     ON DATE(o.order_date_key) = d.full_date
GROUP BY d.year_month, d.year, d.month, c.customer_region
ORDER BY d.year_month, c.customer_region;
--Feeds "Regional Sales Landscape" Dashboard, Monthly Sales by Region line graph

year_month,year,month,customer_region,monthly_revenue,monthly_orders,avg_review_score
2016-09,2016,9,North,136.23,1,1.0
2016-09,2016,9,South,75.06,1,1.0
2016-10,2016,10,Central-West,2857.41,16,3.81
2016-10,2016,10,North,1352.11,5,4.2
2016-10,2016,10,Northeast,7310.45,33,4.39
2016-10,2016,10,South,9767.24,52,3.71
2016-10,2016,10,Southeast,35521.63,202,3.56
2016-12,2016,12,South,19.62,1,5.0
2017-01,2017,1,Central-West,9237.09,43,3.81
2017-01,2017,1,North,5003.28,19,3.84


## Customer Analytics Queries

Supports customer segmentation dashboards including Recency, Frequency, Monetary Value (RFM) analysis, lifetime value, and satisfaction metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE customer_rfm_segmentation AS

WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        c.customer_state,
        c.customer_region,
        o.order_id,
        DATE(o.order_date_key) AS order_purchase_date,
        o.order_total_value,
        o.review_score,
        o.delivered_on_time,
        o.is_delivered,
        o.is_canceled
    FROM gold_fact_orders o
    JOIN gold_dim_customers c ON o.customer_id = c.customer_id
    WHERE o.is_canceled = FALSE
),
dataset_max_date AS (
    SELECT MAX(order_purchase_date) AS max_order_date
    FROM customer_orders
),
customer_metrics AS (
    SELECT
        co.customer_unique_id,
        co.customer_state,
        co.customer_region,
        DATEDIFF(
            d.max_order_date,           
            MAX(co.order_purchase_date)
        ) AS recency_days,
        COUNT(DISTINCT co.order_id)                                     AS total_orders,
        DATEDIFF(
            MAX(co.order_purchase_date),
            MIN(co.order_purchase_date)
        )                                                               AS customer_lifespan_days,
        ROUND(SUM(co.order_total_value), 2)                             AS lifetime_value,
        ROUND(AVG(co.order_total_value), 2)                             AS avg_order_value,
        ROUND(MAX(co.order_total_value), 2)                             AS max_order_value,
        ROUND(AVG(co.review_score), 2)                                  AS avg_review_score,
        SUM(CASE WHEN co.review_score IS NOT NULL THEN 1 ELSE 0 END)    AS reviews_submitted,
        SUM(CASE WHEN co.delivered_on_time THEN 1 ELSE 0 END)           AS on_time_deliveries,
        MIN(co.order_purchase_date)                                     AS first_order_date,
        MAX(co.order_purchase_date)                                     AS last_order_date
    FROM customer_orders co
    CROSS JOIN dataset_max_date d       
    GROUP BY
        co.customer_unique_id,
        co.customer_state,
        co.customer_region,
        d.max_order_date           
),
rfm_scored AS (
-- Score each dimension 1-5 using NTILE quintiles
    SELECT
        *,
        -- Recency: lower days = better = higher score
        6 - NTILE(5) OVER (ORDER BY recency_days DESC)                  AS r_score,
        -- Frequency: higher orders = better = higher score
        NTILE(5) OVER (ORDER BY total_orders ASC)                       AS f_score,
        -- Monetary: higher spend = better = higher score
        NTILE(5) OVER (ORDER BY lifetime_value ASC)                     AS m_score
    FROM customer_metrics
),
rfm_combined AS (
    SELECT
        *,
        -- Combined RFM score (max 15)
        r_score + f_score + m_score                                     AS rfm_total_score,

        -- Engagement health: are high spenders also happy?
        CASE
            WHEN avg_review_score >= 4.0 AND lifetime_value >= 500 THEN 'High Value + Satisfied'
            WHEN avg_review_score < 3.0  AND lifetime_value >= 500 THEN 'High Value + At Risk'
            WHEN avg_review_score >= 4.0 AND lifetime_value < 500  THEN 'Low Value + Satisfied'
            ELSE                                                         'Low Value + Dissatisfied'
        END AS value_sentiment_quadrant,

        -- Review engagement rate
        ROUND(
            reviews_submitted * 100.0
            / NULLIF(total_orders, 0),
        2) AS review_engagement_pct,

        -- On-time delivery rate experienced by this customer
        ROUND(
            on_time_deliveries * 100.0
            / NULLIF(total_orders, 0),
        2) AS personal_on_time_pct
    FROM rfm_scored
),
segmented AS (
    SELECT
        *,
        -- RFM Segment Labels
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyal'
            WHEN r_score >= 4 AND f_score <= 2                   THEN 'Promising'
            WHEN r_score >= 3 AND f_score >= 3 AND m_score <= 2  THEN 'Potential Loyalist'
            WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4  THEN 'At Risk'
            WHEN r_score <= 2 AND f_score >= 3                   THEN 'Needs Attention'
            WHEN r_score = 1  AND f_score = 1                    THEN 'Lost'
            ELSE                                                       'Developing'
        END AS rfm_segment,

        -- Reactivation priority tier
        CASE
            WHEN r_score <= 2 AND m_score >= 4 THEN '1 - Urgent: High Value Gone Quiet'
            WHEN r_score <= 2 AND m_score = 3  THEN '2 - High: Mid Value Gone Quiet'
            WHEN r_score = 3  AND f_score <= 2 THEN '3 - Medium: Infrequent Buyer Cooling'
            WHEN r_score >= 4 AND f_score = 1  THEN '4 - Low: Recent One-Time Buyer'
            ELSE                                    '5 - Monitor'
        END AS reactivation_priority
    FROM rfm_combined
)
SELECT
    customer_unique_id,
    customer_state,
    customer_region,
    first_order_date,
    last_order_date,
    recency_days,
    total_orders,
    customer_lifespan_days,
    lifetime_value,
    avg_order_value,
    max_order_value,
    avg_review_score,
    review_engagement_pct,
    personal_on_time_pct,
    r_score,
    f_score,
    m_score,
    rfm_total_score,
    rfm_segment,
    value_sentiment_quadrant,
    reactivation_priority
FROM segmented
ORDER BY
    reactivation_priority ASC,
    lifetime_value DESC,
    recency_days ASC;
--Feeds "Customer Reactivation Priority" Dashboard, " Customer Concentration by Region" and "Reactivation Priority States" bar charts AND
--"Customer Sentiment and Value Segmentation" Dashboard, "Individual Customer Positioning" scatterplot

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Dependency: requires customer_rfm_segmentation table created by Cell 18
-- Run Cell 18 first before executing this query
SELECT
    reactivation_priority,
    rfm_segment,
    COUNT(*)                            AS customers_in_tier,
    ROUND(SUM(lifetime_value), 2)       AS revenue_at_stake,
    ROUND(AVG(recency_days), 0)         AS avg_days_since_last_order,
    ROUND(AVG(lifetime_value), 2)       AS avg_lifetime_value,
    ROUND(AVG(avg_review_score), 2)     AS avg_satisfaction_score,
    ROUND(AVG(personal_on_time_pct), 2) AS avg_on_time_pct
FROM customer_rfm_segmentation
WHERE reactivation_priority != '5 - Monitor'
GROUP BY reactivation_priority, rfm_segment
ORDER BY reactivation_priority ASC, revenue_at_stake DESC;
--Feeds "Customer Reactivation Priority" Dashboard, "Reactivation Tiers" bar charts AND "Revenue at Stake" donut chart

reactivation_priority,rfm_segment,customers_in_tier,revenue_at_stake,avg_days_since_last_order,avg_lifetime_value,avg_satisfaction_score,avg_on_time_pct
1 - Urgent: High Value Gone Quiet,At Risk,6613,2075806.61,96.0,313.9,4.12,91.79
1 - Urgent: High Value Gone Quiet,Developing,4559,1380594.39,112.0,302.83,4.02,90.06
1 - Urgent: High Value Gone Quiet,Needs Attention,3062,921173.1,96.0,300.84,4.13,91.80
1 - Urgent: High Value Gone Quiet,Lost,1465,447287.01,51.0,305.32,4.17,95.70
2 - High: Mid Value Gone Quiet,Needs Attention,4582,497107.76,95.0,108.49,4.22,92.99
2 - High: Mid Value Gone Quiet,Developing,2343,254044.24,111.0,108.43,4.19,90.74
2 - High: Mid Value Gone Quiet,Lost,782,84929.73,51.0,108.61,4.35,96.04
3 - Medium: Infrequent Buyer Cooling,Developing,7666,1172655.77,225.0,152.97,3.98,88.77
4 - Low: Recent One-Time Buyer,Promising,7657,1249392.55,401.0,163.17,4.11,92.31


In [0]:
%sql
SELECT
    CASE
        WHEN lifetime_value >= 500 THEN 'High Value'
        ELSE 'Low Value'
    END                                         AS value_tier,
    CASE
        WHEN avg_review_score >= 4.0 THEN 'Satisfied (≥4.0)'
        ELSE 'At Risk / Dissatisfied (<4.0)'
    END                                         AS sentiment_tier,
    COUNT(*)                                    AS customer_count,
    ROUND(SUM(lifetime_value), 2)               AS total_lifetime_value,
    ROUND(AVG(avg_review_score), 2)             AS avg_review_score,
    ROUND(AVG(recency_days), 0)                 AS avg_recency_days,
    ROUND(AVG(total_orders), 1)                 AS avg_orders
FROM customer_rfm_segmentation
GROUP BY
    CASE WHEN lifetime_value >= 500 THEN 'High Value' ELSE 'Low Value' END,
    CASE WHEN avg_review_score >= 4.0 THEN 'Satisfied (≥4.0)' ELSE 'At Risk / Dissatisfied (<4.0)' END
ORDER BY total_lifetime_value DESC;
--Feeds "Customer Sentiment and Value Segmentation" Dashboard, Value Sentiment Quadrant

value_tier,sentiment_tier,customer_count,total_lifetime_value,avg_review_score,avg_recency_days,avg_orders
Low Value,Satisfied (≥4.0),70099,8882924.08,4.75,242.0,1.0
High Value,Satisfied (≥4.0),3157,2900320.33,4.76,242.0,1.1
Low Value,At Risk / Dissatisfied (<4.0),20544,2812814.83,1.92,249.0,1.0
High Value,At Risk / Dissatisfied (<4.0),1226,1141464.82,1.75,240.0,1.1


In [0]:
%sql
SELECT
    FLOOR(avg_review_score) AS score_bucket,
    CASE
        WHEN lifetime_value >= 500 THEN 'High Value'
        ELSE 'Low Value'
        END AS value_tier,
    COUNT(*) AS customer_count
FROM customer_rfm_segmentation
WHERE avg_review_score IS NOT NULL
GROUP BY FLOOR(avg_review_score), value_tier
ORDER BY score_bucket, value_tier;
--Feeds "Customer Sentiment and Value Segmentation" Dashboard, "Satisfaction Score Distribution" histogram

score_bucket,value_tier,customer_count
1,High Value,670
1,Low Value,9347
2,High Value,161
2,Low Value,2888
3,High Value,351
3,Low Value,7668
4,High Value,787
4,Low Value,17888
5,High Value,2370
5,Low Value,52211
